# VietAI/vit5-base — model exploratory

**Three questions**
1. What is this model made of, and where would an embedding map attach?
2. How does it cut a Vietnamese/English dish name into tokens?
3. With no fine-tuning, can it copy a sentence — and where does it hesitate?

Pairing rule from the mission: this model is compared **base with base** (`flan-t5-base`), never across sizes.

In [1]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


python 3.12.10 | torch 2.7.1+cu128 | cuda True | NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Load — and print the device

`res-33`: a run that 'used the GPU' is a claim until the device string is printed.

In [2]:
tok, model, info = M.load("vit5-base")
info

loaded vit5-base        226.0M params  d_model=768 enc/dec=12/12  device=cuda dtype=bfloat16 vram=448 MB  4.39s


{'model_key': 'vit5-base',
 'hf_id': 'VietAI/vit5-base',
 'd_model': 768,
 'layers_enc': 12,
 'layers_dec': 12,
 'heads': 12,
 'vocab': 36096,
 'params': 225950976,
 'dtype': 'bfloat16',
 'device': 'cuda',
 'load_s': 4.39,
 'vram_mb': 447.6,
 'pad_id': 0,
 'eos_id': 1}

## 2. Architecture — the surface a mapping would touch

In [3]:
cfg = model.config
rows = [("d_model", cfg.d_model), ("encoder layers", cfg.num_layers),
        ("decoder layers", cfg.num_decoder_layers), ("heads", cfg.num_heads),
        ("d_ff", cfg.d_ff), ("vocab", cfg.vocab_size),
        ("relative buckets", cfg.relative_attention_num_buckets), ("max positions", cfg.n_positions)]
enc = sum(p.numel() for p in model.encoder.parameters())
dec = sum(p.numel() for p in model.decoder.parameters())
emb = model.shared.weight.numel()
print(f"{'property':<20}{'value':>12}")
for k, v in rows: print(f"{k:<20}{v:>12}")
print(f"{'params enc':<20}{enc/1e6:>10.1f}M")
print(f"{'params dec':<20}{dec/1e6:>10.1f}M")
print(f"{'params emb (shared)':<20}{emb/1e6:>10.1f}M")

property                   value
d_model                      768
encoder layers                12
decoder layers                12
heads                         12
d_ff                        3072
vocab                      36096
relative buckets              32
max positions                512
params enc               112.7M
params dec               141.0M
params emb (shared)       27.7M


**Read this as:** every encoder layer outputs a `[seq, 768]` matrix. The paper's map
sits between two sequences of such matrices, so the number of *candidate attach points* is
`layers + 1` per side, and the two models must agree on `d_model` (768 → 768).

## 3. Tokenizer — a dish name is a span, not a token

In [4]:
samples = D.build(force=False)
lang = C.MODELS["vit5-base"]["lang"]
texts = [r[lang] for r in samples]
dishes = [r["dish_vi"] if lang == "vi" else r["dish_en"] for r in samples]

tpw = [len(tok(t, add_special_tokens=False)["input_ids"]) / max(1, len(t.split())) for t in texts]
dtok = [len(tok(d, add_special_tokens=False)["input_ids"]) for d in dishes]
print(f"tokens per word      : mean {sum(tpw)/len(tpw):.2f}  max {max(tpw):.2f}")
print(f"dish-name subwords   : mean {sum(dtok)/len(dtok):.2f}  max {max(dtok)}")

ids = tok('Hôm nay tôi ăn phở bò với bánh mì ở quán quen.', add_special_tokens=False)["input_ids"]
print("\nsentence:", 'Hôm nay tôi ăn phở bò với bánh mì ở quán quen.')
print("pieces  :", " | ".join(p.replace("\u2581", "\u2423") for p in tok.convert_ids_to_tokens(ids)))

OK  100 pairs | csv -> data\culinary_100_pairs.csv
    vi chars 5540 | en chars 6997
tokens per word      : mean 1.19  max 1.82
dish-name subwords   : mean 3.03  max 6

sentence: Hôm nay tôi ăn phở bò với bánh mì ở quán quen.
pieces  : ␣Hôm | ␣nay | ␣tôi | ␣ăn | ␣phở | ␣bò | ␣với | ␣bánh | ␣mì | ␣ở | ␣quán | ␣quen | .


Vietnamese words are split more finely than English ones (diacritics + a VI vocabulary),
so a dish name can be 1–6 subword pieces. **Consequence for measurement:** "perplexity of the dish name"
must be computed over a token *span*, not a single token — which is what `src/score.py` does.

## 4. One forward pass — what the layers actually hold

In [5]:
enc_in = tok(['Hôm nay tôi ăn phở bò với bánh mì ở quán quen.'], return_tensors="pt").to(M.device())
with torch.no_grad():
    enc_out = model.encoder(**enc_in, output_hidden_states=True)
hs = enc_out.hidden_states
print("hidden states returned:", len(hs), "| shape:", tuple(hs[0].shape))
for i, h in enumerate(hs):
    print(f"layer {i:2d}  pooled-mean norm {h[0].mean(0).norm():7.2f}  last-token norm {h[0][-1].norm():7.2f}")

hidden states returned: 13 | shape: (1, 14, 768)
layer  0  pooled-mean norm   13.69  last-token norm   59.00
layer  1  pooled-mean norm   48.50  last-token norm  110.50
layer  2  pooled-mean norm   62.00  last-token norm  138.00
layer  3  pooled-mean norm   72.50  last-token norm  156.00
layer  4  pooled-mean norm   85.00  last-token norm  163.00
layer  5  pooled-mean norm   96.50  last-token norm  170.00
layer  6  pooled-mean norm  111.00  last-token norm  185.00
layer  7  pooled-mean norm  129.00  last-token norm  198.00
layer  8  pooled-mean norm  157.00  last-token norm  222.00
layer  9  pooled-mean norm  190.00  last-token norm  236.00
layer 10  pooled-mean norm  220.00  last-token norm  270.00
layer 11  pooled-mean norm  258.00  last-token norm  314.00
layer 12  pooled-mean norm   15.75  last-token norm   22.62


The norm grows with depth, and the last token of the encoder is the summary a decoder would
attend to. If you later probe "which layer carries the dish meaning", these 13 matrices are the grid.

## 5. Zero-shot behaviour (no fine-tuning)

In [6]:
def gen(prompt, n=64):
    ids = tok(prompt, return_tensors="pt").to(M.device())
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, num_beams=1)
    return tok.decode(out[0], skip_special_tokens=True)

for p in ['Nói: Phở bò là món ăn nổi tiếng của Việt Nam.', 'translate English to Vietnamese: Vietnamese food is rich and varied.', '1 + 3 = ?']:
    print("PROMPT:", p)
    print("OUTPUT:", gen(p))
    print()

PROMPT: Nói: Phở bò là món ăn nổi tiếng của Việt Nam.


OUTPUT: *:+._ CóZ Có[:]:!!> Cój:&:W:Ỵ.Ẽ:Ỡ:Ẫ:Ẳ:Ằ:Õ.Ẻ:Ặ:Ỗ:Ẹ CóỸ:È.Ữ:Ự.Ợ:Ử. Phở bò là món ăn nổi tiếng

PROMPT: translate English to Vietnamese: Vietnamese food is rich and varied.


OUTPUT: *+__Z_Z[]!>j&.W andỴ to andẼ to to andỠ to toẪ inẲ toẰ inÕ inẺ inẶ toỖ toẸ is to andỸÈỮỮỰỢỢ toỬ. Xem thêm: Vietnamese food is rich

PROMPT: 1 + 3 = ?


OUTPUT: * + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ? + 3 + 3  ?



## 6. Copy test — 2,000 sentences

In [7]:
rows = read_csv("copy_vit5-base.csv")
if rows:
    bad = [r for r in rows if r["exact"] == "0"][:5]
    print(f"exact copies: {sum(r['exact'] == '1' for r in rows)} / {len(rows)}")
    for r in bad:
        print("SRC:", r["src"][:110])
        print("HYP:", r["hyp"][:110])
        print()
    s = [r for r in read_csv("copy_summary.csv") if r["model"] == "vit5-base"]
    if s:
        print({k: s[0][k] for k in ("n", "exact_match", "exact_match_norm", "bleu", "chrf",
                                   "tok_recall", "mean_hyp_words", "mean_ref_words")})

exact copies: 0 / 2000
SRC: Cái khát vọng của người phụ nữ có cho sự phát triển của bản thân là thứ quan trọng nhất .
HYP: *+__Z______________________________________________________________________Z__

SRC: Nhận thức được sự huỷ hoại và những thách thức , tôi cũng như bao nhiêu phụ nữ khác , hăm hở tái xây dựng xã h
HYP: *+_Z nói:[]]!> Nói:j nói:& nói: Tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi, tôi

SRC: Như các bạn thấy đó , các bóng đèn hướng ra phía ngoài , bởi vì đó là hướng sư tử tiến vào .
HYP: * các bạn thấy đó, các bóng đèn hướng ra phía ngoài .... Các bạn thấy đó, các bóng đèn hướng ra phía ngoài ...

SRC: Tôi đứng trên sân khấu này bởi vì tôi xinh đẹp , một phụ nữ da trắng , và trong ngành người mẫu , chúng tôi gọ
HYP: *, và+__ZZ______________________________________________________________________

SRC: Từ đó chúng tôi bắt đầu nhờ khách du lịch qua đường chụp ảnh
HYP: * chụp ảnh+___Z chụp ảnh____Z chụp ảnh[]]! Chúng tôi! Chúng tôi bắt đầu đi 

## 7. Perplexity — is the dish name harder than the rest of the sentence?

In [8]:
s = [r for r in read_csv("ppl_summary.csv") if r["model"] == "vit5-base"]
if s:
    r = s[0]
    print({k: r[k] for k in ("ppl_all", "ppl_dish", "ppl_other", "rank_dish", "rank_other",
                            "delta_nll", "delta_ci_lo", "delta_ci_hi", "dish_harder_sentences", "sentences")})
toks = read_csv("ppl_tokens_vit5-base.csv")
if toks:
    print("\nhardest tokens (highest NLL):")
    for r in sorted(toks, key=lambda x: -float(x["nll"]))[:10]:
        print(f"  {float(r['nll']):6.2f}  rank {int(r['rank']):>6}  dish={r['in_dish']}  {r['span']!r}")

{'ppl_all': '74.21', 'ppl_dish': '2803425.14', 'ppl_other': '10.03', 'rank_dish': '1148.2', 'rank_other': '27.2', 'delta_nll': '14.468', 'delta_ci_lo': '12.855', 'delta_ci_hi': '16.163', 'dish_harder_sentences': '96', 'sentences': '100'}

hardest tokens (highest NLL):
   38.97  rank  24431  dish=1  'nitz'
   35.19  rank  10553  dish=1  'ỏi'
   34.81  rank  21038  dish=1  'Pad'
   34.62  rank   4837  dish=1  'ẩu'
   34.44  rank   6469  dish=1  'um'
   34.17  rank  18905  dish=1  'Tir'
   33.06  rank   9837  dish=1  'ỏi'
   32.97  rank  10845  dish=1  'lava'
   32.75  rank   5523  dish=1  'ẩu'
   32.73  rank  10616  dish=1  'Mì'


`delta_nll > 0` means dish tokens are *less* predictable than the sentence around them.
The CI is a paired bootstrap over the 100 sentences, so it answers "is the dish harder **in the same sentence**",
not "is this model worse than that model".

## 8. What this means for the paper

- The *copy gate* is answered in §6: if `exact_match` is low, fine-tuning is not optional — the model will not
  even reproduce its own input, so a mapped vector cannot be read out.
- The *attach surface* in §2/§4 is `[seq, 768]` at 13 depths; the pairing rule keeps `base` with `base`.
- The *dish span* in §3/§7 is the unit the paper's perplexity story has to be told in.